# 🌟 Layer 1: Universal Omni-Channel Data Ingestion (v4)

---

## Executive Summary

Welcome to the **Layer 1 Ingestion Pipeline** for Intelligent AML. 

The primary goal of Layer 1 is to seamlessly read heavily-fragmented, heterogeneous financial datasets (ranging from traditional fiat to cryptocurrency tokens) and map them into a **Unified Graph Topology**. In earlier iterations, this was attempted strictly via in-memory tools like `Polars` and `Pandas`. However, loading datasets like *Elliptic v2* (nearly 200 million edges) natively into memory often led to catastrophic `No space left on device` crashes.

### Why V4 is a Breakthrough
In this V4 architecture, we shift the heavy lifting to **DuckDB**.
- **Bounded Memory Footprint:** DuckDB enforces strict RAM limits, automatically spilling to a temporary disk (`TEMP_DIR`) rather than crashing.
- **Schema Agnosticism:** We utilize advanced column sniffing (`read_csv_auto()`) to dynamically identify identifiers (`src`, `dst`) and labels across any dataset, bypassing hard-coded schemas.
- **Parquet & ZSTD Compression:** All outputs are stored as chunked Parquet files with ZSTD compression, yielding massive storage savings while allowing fast, predicate-pushdown reads for Layer 2 GNN training.
- **Modular and Web-Ready:** The core logic now lives in `src.ingestion.pipeline`, enabling this exact pipeline to be triggered programmatically by our Web Backend.

---
## 1. Initializing the Engine

We begin by importing our centralized pipeline. This pipeline abstracts away the tedious setup of DuckDB connection strings, limits, and workspace paths.

> **Note:** This notebook acts as an interactive execution interface. The robust backend runs natively on Python for maximum performance.

In [ ]:
import sys
from pathlib import Path

# Ensure our root project directory is discoverable so we can import the src module
root_dir = str(Path().resolve().parent.parent)
if root_dir not in sys.path:
    sys.path.append(root_dir)

import polars as pl
import duckdb
from src.ingestion.pipeline import *

print("\u2705 Intelligent AML Pipeline Module successfully loaded!")
print(f"\U0001f4ca DuckDB Engine limits enforced: {DUCKDB_MEMORY_LIMIT_GB} GB RAM allocated.")

---
## 2. Dynamic Memory Management & Safety Checks

Before we execute, it is crucial to understand *how* the pipeline manages risk.

We wrap dataset parsing functions in a higher-order wrapper called `run_safely`. 

### Why do we need `run_safely`?
If one node or dataset in a federated learning context fails (e.g., due to corrupt formatting or abrupt resource exhaustion), it should **never** halt the entire pipeline. `run_safely` guarantees fault isolation, allowing us to ingest 18 distinct datasets asynchronously without worrying about a single corrupted CSV taking the framework offline.

In [ ]:
# Example of checking pre-flight resources
report_resources(tag="pre-flight-check")

---
## 3. The Grand Execution: Traversing Heterogeneous Domains

Intelligent AML aims to recognize illicit patterns universally. To achieve this, the pipeline ingests data across several modules:
1. **Traditional Fiat:** Standard baseline datasets like PaySim and synthesized card transactions.
2. **Crypto & Web3:** Real-world adversarial networks such as Mt.Gox Leaks, Ethereum Phishing graphs, and the monumental Elliptic v1 & v2 sets.
3. **Specialized IBM AML Patterns:** Deeply complex multi-tiered laundering scheme simulations.

By calling `run_all_datasets()`, we command DuckDB to sweep through every domain, mapping flat CSV files and Python Pickles into graph-ready `nodes.parquet` and `edges.parquet` topologies.

In [ ]:
# Warning: Running this executes the entire Layer 1 pipeline. 
# Ensure you have mounted the required Kaggle input directories.

# run_all_datasets()  # <-- Uncomment to execute the master ingestion

---
## 4. Incremental / Streaming Data Ingestion

Financial systems are not static. To reflect real-time production settings, our pipeline natively supports **Live Feed Checkpointing**.

When a new transaction block arrives (e.g., via Kafka, Websockets, or API poller), we use `ingest_stream_batch` to append it. The engine automatically manages deduplication using a checkpoint registry (`_checkpoints.parquet`).

Below is a demonstration of how this handles continuous streams seamlessly.

In [ ]:
print("=== Simulated Live Feed Demonstration ===")

# We simulate 3 small batches of incoming transaction records
for batch_id, batch in simulate_live_feed(n_batches=3, records_per_batch=2):
    result = ingest_stream_batch("live_demo", batch, batch_id=batch_id)
    print(f"  {batch_id}: {result['status']}, rows={result.get('rows')}")

unified = load_full_dataset("live_demo")
if unified is not None:
    print(f"\n\u2705 Unified view compiled dynamically: {len(unified):,} rows in total")
else:
    print("\n\u274c Failed to compile unified dataset.")

print("\n=== Re-running identical feed to prove deduplication ===")
for batch_id, batch in simulate_live_feed(n_batches=3, records_per_batch=2):
    result = ingest_stream_batch("live_demo", batch, batch_id=batch_id)
    print(f"  {batch_id}: {result['status']}")

---
## 🏁 Hand-off to Layer 2

Once the ingestion completes, your `graph_data/` directory will be heavily populated with `nodes.parquet` and `edges.parquet` subsets.

These files are structurally perfected for **Layer 2 (The HT-GNN Detection Engine)**. Because the data has been transformed universally into graph semantics (edges representing transactions, nodes representing accounts), Layer 2 can seamlessly traverse domains without manual data wrangling per dataset.

> **Ready for Exploration?** Head over to `notebooks/00_EDA/01_Layer1_EDA.ipynb` to visualize the mathematical distribution of these generated topologies!